# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshithavandavasi-1401/HV-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
print(os.getcwd())

/content


In [6]:
!git clone https://github.com/harshithavandavasi-1401/HV-flyrank-ml-internship.git

Cloning into 'HV-flyrank-ml-internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 52), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.87 MiB | 11.79 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [7]:
%cd /content/HV-flyrank-ml-internship

/content/HV-flyrank-ml-internship


In [8]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [10]:
import pandas as pd
import numpy as np
import os

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

(30000, 44)


## 1. My rule and its reason codes

### My Rule

I rank pages for refresh using historical SEO signals that are available before prediction.

The rule gives higher scores to pages that:

- have not been updated recently,
- have a low click-through rate,
- already rank reasonably well in search results.

These pages are more likely to benefit from content improvements.

Reason codes:

- STALE_CONTENT
- LOW_CTR
- SEO_OPPORTUNITY
- HEALTHY

In [11]:
reason_codes = [
    "STALE_CONTENT",
    "LOW_CTR",
    "SEO_OPPORTUNITY",
    "HEALTHY"
]

print("Reason Codes")
for code in reason_codes:
    print("-", code)

Reason Codes
- STALE_CONTENT
- LOW_CTR
- SEO_OPPORTUNITY
- HEALTHY


## 2. Build the ranked queue (writes the CSV)

### Ranked Queue

An Action Score is assigned using three historical signals.

Scoring rule:

- +40 if the page has not been updated for more than 180 days.
- +30 if CTR is below the dataset median.
- +30 if average search position is better than the dataset median.

The total score ranges from 0 to 100.

Higher scores indicate higher priority for review.

In [12]:
baseline = df.copy()

baseline["action_score"] = 0
baseline["reason_code"] = ""
baseline["action"] = "Monitor"

ctr_threshold = baseline["ctr"].median()
position_threshold = baseline["avg_position"].median()

# Stale content
mask = baseline["days_since_last_update"] > 180
baseline.loc[mask, "action_score"] += 40
baseline.loc[mask, "reason_code"] += "STALE_CONTENT;"

# Low CTR
mask = baseline["ctr"] < ctr_threshold
baseline.loc[mask, "action_score"] += 30
baseline.loc[mask, "reason_code"] += "LOW_CTR;"

# Good ranking opportunity
mask = baseline["avg_position"] < position_threshold
baseline.loc[mask, "action_score"] += 30
baseline.loc[mask, "reason_code"] += "SEO_OPPORTUNITY;"

baseline.loc[
    baseline["reason_code"] == "",
    "reason_code"
] = "HEALTHY"

baseline.loc[
    baseline["action_score"] >= 70,
    "action"
] = "Refresh Immediately"

baseline.loc[
    (baseline["action_score"] >= 40) &
    (baseline["action_score"] < 70),
    "action"
] = "Review"

baseline = baseline.sort_values(
    "action_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows:", len(baseline))
print("CSV written successfully.")

baseline[
    [
        "content_id",
        "action_score",
        "reason_code",
        "action"
    ]
].head(10)

Rows: 30000
CSV written successfully.


,content_id,action_score,reason_code,action
6533,content_9b7a283bd1c9,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
25571,content_5262fa59b6bc,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
2519,content_0edf498ae135,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
15243,content_10bd2440e4ac,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
7719,content_f783292bc4a0,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
20824,content_da54cb4484f1,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
8709,content_77834a072fd3,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
5003,content_17aa56bdad68,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
21575,content_be9a0395c1e4,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately
24603,content_06f581dd9383,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately


## 3. Top-20 review

The highest-ranked pages were reviewed manually.

These pages were selected because they combine stale content, low CTR, or a good ranking opportunity.

The recommendations should be treated as decision support rather than automatic actions because additional business context may affect the final decision.

In [13]:
top20 = baseline.head(20).copy()

top20["confidence"] = np.where(
    top20["action_score"] >= 70,
    "High",
    "Medium"
)

top20["what_would_make_it_wrong"] = (
    "Seasonal content, recent unpublished updates, or low search demand."
)

top20[
    [
        "content_id",
        "action_score",
        "reason_code",
        "action",
        "confidence",
        "what_would_make_it_wrong"
    ]
]

,content_id,action_score,reason_code,action,confidence,what_would_make_it_wrong
6533,content_9b7a283bd1c9,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
25571,content_5262fa59b6bc,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
2519,content_0edf498ae135,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
15243,content_10bd2440e4ac,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
7719,content_f783292bc4a0,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
20824,content_da54cb4484f1,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
8709,content_77834a072fd3,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
5003,content_17aa56bdad68,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
21575,content_be9a0395c1e4,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."
24603,content_06f581dd9383,100,STALE_CONTENT;LOW_CTR;SEO_OPPORTUNITY;,Refresh Immediately,High,"Seasonal content, recent unpublished updates, ..."


## 4. Weak picks + leakage check

### Weak Picks

This rule may incorrectly prioritize:

- seasonal content,
- pages with naturally low search demand,
- pages that were updated recently but have not yet shown performance improvements,
- pages that require technical SEO fixes instead of content updates.

### Leakage Check

Only historical features were used to calculate the Action Score.

The following columns were intentionally excluded:

- trend_direction
- trend_pct

These columns contain future outcome information and would introduce target leakage.

In [14]:
excluded = [
    "trend_direction",
    "trend_pct"
]

print("Excluded Columns")

for column in excluded:
    print("-", column)

print("\nLeakage Check: PASSED")
print("Only historical features were used.")

Excluded Columns
- trend_direction
- trend_pct

Leakage Check: PASSED
Only historical features were used.


## Self-check

- ✅ Explained the baseline rule.
- ✅ Defined reason codes.
- ✅ Built a ranked queue.
- ✅ Generated `baseline_action_score.csv`.
- ✅ Reviewed the top 20 recommendations.
- ✅ Discussed weak picks.
- ✅ Confirmed that no future information or target labels were used.
- ✅ Notebook runs from top to bottom without errors.